In this notebook, we implement a baseline corresponding to the model proposed by [Frey et al (2019)](https://pubs.acs.org/doi/abs/10.1021/acsnano.8b08014) using PU learning. Simply put, this is a PU bagging where the base estimator has been replaced by a decision tree.

To be fair with their setup, we perform some grid search over different bag sizes and K-fold validation.

The original code of Frey et al can be found in this [repo](https://github.com/ncfrey/pumml/blob/master/pumml/learners.py)

In [1]:
import pandas as pd
import pulearn
from sklearn import preprocessing, model_selection, metrics, tree
import numpy as np
from tqdm import tqdm
import itertools
import os

In [2]:
# We first begin by the same preprocessing that we applied to our dataset for the clustering models
X_cont = pd.read_csv("../data/continuous_data.csv", index_col=0)
X_cont = preprocessing.MinMaxScaler().fit_transform(X_cont)
X_cat = pd.read_csv("../data/categorical_data.csv", index_col=0).to_numpy()
X = np.concatenate([X_cont, X_cat], axis=1)
pu_labels = pd.read_csv("../data/pu_labels.csv", index_col=0)
y = (pu_labels["class"]>0).to_numpy().astype(float)

In [3]:
def pu_score(y_true, y_pred):
    """Computes the pseudo-F1 score of PU learning.
    This assumes that observing labels is independent of data value
    """
    return metrics.recall_score(y_true, y_pred)**2 / y_pred.mean()

Let's try to run the same thing using only MXenes that were discovered at the time of publication of Frey et al. This means before 2019.

According to the paper, they had 10 MXenes. We use:

|MAX|MX|Year|
|:---:|:---:|:---:|
|Ti2AlC|Ti2C|2012|
|V2AlC|V2C|2013|
|Nb2AlC|Nb2C|2013|
|Ti2AlN|Ti2N|2017|
|Ti3AlC2|Ti3C2|2011|
|Ti3SiC2|Ti3C2|2018|
|Ta4AlC3|Ta4C3|2012|
|Nb4AlC3|Nb4C3|2014|
|V4AlC3|V4C3|2018|
|Ti4AlN3|Ti4N3|2016|

In [4]:
X_cont = pd.read_csv("../data/continuous_data.csv", index_col=0)
X_cont = preprocessing.MinMaxScaler().fit_transform(X_cont)
X_cat = pd.read_csv("../data/categorical_data.csv", index_col=0).to_numpy()
X = np.concatenate([X_cont, X_cat], axis=1)
pu_labels = pd.read_csv("../data/pu_labels.csv", index_col=0)
mxenes_2019 = ["Ti2AlC", "V2AlC", "Nb2AlC", "Ti2AlN", "Ti3AlC2", "Ti3SiC2", "Ta4AlC3", "Nb4AlC3", "V4AlC3", "Ti4AlN3"]
print(len(mxenes_2019))
pu_labels["class"] = 0
pu_labels.loc[mxenes_2019, "class"] = 1
y = pu_labels["class"].to_numpy().astype(int)

10


In [5]:
k_params =  [2, 3, 5, 10]
T_params = [10, 50, 100, 200, 500, 1000]

R = 200 # Number of repeats (to get sub-estimates for lower R, subsampling would suffice)

feature_indices=[f"feature_{i}" for i in range(X.shape[1])]
sample_indices=[f"max_{i}" for i in range(len(X))]

for (i,k),(j,T) in itertools.product(enumerate(k_params), enumerate(T_params)):

    csv_filename = os.path.join("baseline_2019",f"fold_{k}_estimators_{T}.csv") 
    if os.path.exists(csv_filename):
        continue

    splitter = model_selection.RepeatedStratifiedKFold(n_splits=k, n_repeats=R, random_state=0)

    all_outputs = []

    for r, (train_indices, test_indices) in enumerate(tqdm(splitter.split(X,y))):
        base_estimator = tree.DecisionTreeClassifier(class_weight="balanced")
        pu_estimator = pulearn.BaggingPuClassifier(estimator=base_estimator, n_estimators=T, max_samples=k, max_features=X.shape[1], n_jobs=8)
    
        # Fit
        pu_estimator.fit(X[train_indices], y[train_indices])
    
        # We now must get the predictions on the test set and the training set
        y_pred = np.zeros(len(y))
        y_pred[train_indices] = pu_estimator.oob_decision_function_[:,1]
        y_pred[test_indices] = pu_estimator.predict_proba(X[test_indices])[:,1]
    
        # Now, we must estimate the recall on the test set, and the PU score
        tpr = metrics.recall_score(y[test_indices], y_pred[test_indices].round())
        pu_score = tpr**2 / (1e-12+y_pred[test_indices].round().mean())
    
        feature_importances = sum([x.feature_importances_ for x in pu_estimator.estimators_])

        feature_importances = pd.Series(data=feature_importances,index=feature_indices)
        probs = pd.Series(data=y_pred, index=sample_indices)
        data_row = pd.concat([feature_importances, probs])
        data_row["Split"] = r%k
        data_row["Repeat"] = r//k
        data_row["TPR"] = tpr
        data_row["PU_score"] = pu_score

        all_outputs += [data_row]

    pd.DataFrame(all_outputs).to_csv(csv_filename)
        